# Study 881 — Jobless-Claims Sector Rotation 📋

**Does a rise in initial jobless claims tilt the market from cyclicals to defensives?**

The nowcast folk-rule: rising **initial jobless claims** signal a cooling labour market,
so the risk-off playbook says rotate from **cyclicals** (XLY consumer-discretionary,
XLI industrials) into **defensives** (XLP staples, XLU utilities). We test the sharp
version — does the **4-week change in claims** *predict* the forward
**cyclical-minus-defensive** sector spread? — on a monthly frame (1998-12-31 →
2026-06-30, 331 months). This is a *rotation*, not a market-timer (that is
Study 385's question).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. The sector ETFs trade continuously since 1998 — no survivorship in
the outcome; the one honest hazard is the 2020 outlier.*


## 1. The idea in one line

Claims come out weekly and lead the cycle; defensives out-earn into slowdowns. So a claims *uptick* should push the **cyclical − defensive** return spread **down** next month. That means a **negative** predictive slope. Let's see what the tape says.

In [1]:
R = {'start': '1998-12-31', 'end': '2026-06-30', 'n_months': 331, 'n': 329, 'fingerprint': '43e0a9ec1a15', 'spread_mean_pct': 0.251, 'spread_sd_pct': 4.37, 'slope': 0.0177, 't_nw': 6.39, 'r2': 0.0127, 'corr': 0.1125, 'ex_covid_slope': 0.0623, 'ex_covid_t': 1.49, 'ex_covid_n': 318, 'winsor_slope': 0.0504, 'winsor_t': 1.43, 'spearman_rho': 0.0189, 'spearman_p': 0.733, 'era_early_slope': 0.1222, 'era_early_t': 1.41, 'era_early_n': 156, 'era_late_slope': 0.0164, 'era_late_t': 6.83, 'era_late_n': 173, 'placebo_obs': 0.0177, 'placebo_mean': -3e-05, 'placebo_sd': 0.0089, 'placebo_p': 0.053, 'timer0_gross': 0.89, 'timer0_net': 0.39, 'timer0_t': 0.14, 'timer10_gross': 0.89, 'timer10_net': -1.91, 'timer10_t': -0.66, 'n_switches': 168, 'null_mean_t': 0.16, 'null_sd_t': 0.95, 'null_fire': 1, 'planted_slope': -0.4651, 'planted_t': -13.85}
print('predictive slope of (cyclical - defensive) spread on the 4-week claims change:')
print('  slope %+.4f   NW(6) t = %+.2f   R2 = %.4f   (n=%d)' % (R['slope'], R['t_nw'], R['r2'], R['n']))
need = 'negative'; got = 'NEGATIVE' if R['slope'] < 0 else 'POSITIVE (wrong sign!)'
print('  claim needs a %s slope; observed is %s' % (need, got))

predictive slope of (cyclical - defensive) spread on the 4-week claims change:
  slope +0.0177   NW(6) t = +6.39   R2 = 0.0127   (n=329)
  claim needs a negative slope; observed is POSITIVE (wrong sign!)


## 2. The catch — it is *one* month (2020)

The fitted slope is not just wrong-signed, it is a mirage of a single episode. In 2020 the claims 4-week MA exploded (211k → 4,174k) exactly as the market bottomed and cyclicals ripped off that bottom — a few enormous same-signed points that drag the regression line.

In [2]:
print('full sample : slope %+.4f   NW t = %+.2f' % (R['slope'], R['t_nw']))
print('ex-COVID 2020: slope %+.4f   NW t = %+.2f  (n=%d)' % (R['ex_covid_slope'], R['ex_covid_t'], R['ex_covid_n']))
print('winsor 1/99  : slope %+.4f   NW t = %+.2f' % (R['winsor_slope'], R['winsor_t']))
print('Spearman rank: rho %+.4f   p = %.3f  (outlier-robust -> ~zero)' % (R['spearman_rho'], R['spearman_p']))

full sample : slope +0.0177   NW t = +6.39
ex-COVID 2020: slope +0.0623   NW t = +1.49  (n=318)
winsor 1/99  : slope +0.0504   NW t = +1.43
Spearman rank: rho +0.0189   p = 0.733  (outlier-robust -> ~zero)


## 3. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`: rising claims really do knock cyclicals down) and check the detector recovers the **negative** slope — and stays *silent* on the null (`edge=0`, claims move but predict nothing). No network.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from claims_nowcast import data, strategy as st
null = st.synthetic_detect(data.synthetic_frame(edge=0.0, seed=881, n_months=360))
planted = st.synthetic_detect(data.synthetic_frame(edge=0.5, seed=881, n_months=360))
print('null world   : slope NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: slope NW t = %+.2f  (should light up NEGATIVE)' % planted['t_nw'])

null world   : slope NW t = +1.04  (should be ~0)
planted world: slope NW t = -13.85  (should light up NEGATIVE)


## 4. The honest verdict

On the real tape the predictive slope is **wrong-signed and significant** (**+0.0177**, NW *t* = **+6.39**) — but that significance is a **single-outlier (COVID-2020)** artefact: it collapses to *t* ≈ 1.5 once 2020 is dropped, to a Spearman ρ = +0.02 (p = 0.73), and to *t* = +1.41 in the pre-2020 era. The seeded control recovers a *planted* rotation cleanly, so the engine is fine — the effect simply is not there. Rising claims carry **no** robust cyclical-vs-defensive rotation signal. **Signal: None. Tradability: Mirage.**